In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.ops as ops

# =====================================================================
# 1. MODEL ARCHITECTURE: TWO-STAGE ROI DETECTOR FROM SCRATCH
# =====================================================================


class TwoStageRoIDetector(nn.Module):
    def __init__(self, num_classes, pool_size=7):
        super().__init__()
        self.num_classes = num_classes  # Excluding background class
        self.pool_size = pool_size

        # Using allowed torchvision backbone (ResNet-34)
        # Extracting feature maps prior to adaptive pooling/fully-connected layers
        resnet = models.resnet34(weights=None)
        self.backbone = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,  # [B, 64, H/4, W/4]
            resnet.layer2,  # [B, 128, H/8, W/8]
            resnet.layer3,  # [B, 256, H/16, W/16]
            resnet.layer4,  # [B, 512, H/32, W/32] -> Final Feature Map
        )

        # Spatial downsampling ratio from original image space to the backbone feature map
        # ResNet-34 downsamples the input by a factor of 32 (2^5)
        self.spatial_scale = 1.0 / 32.0

        # Head Feature Extractor
        in_channels = 512
        mid_channels = 1024
        self.box_head = nn.Sequential(
            nn.Linear(in_channels * pool_size * pool_size, mid_channels),
            nn.ReLU(inplace=True),
            nn.Linear(mid_channels, mid_channels),
            nn.ReLU(inplace=True),
        )

        # Predictor heads:
        # Classification output maps to (num_classes + 1) to incorporate the background token
        self.class_predictor = nn.Linear(mid_channels, num_classes + 1)
        # Regression output assigns scale-invariant coordinate offsets independently for each class
        self.bbox_predictor = nn.Linear(mid_channels, (num_classes + 1) * 4)

    def forward(self, images, rpn_proposals):
        """
        Args:
            images (Tensor): Batch of images matching shape (B, C, H, W)
            rpn_proposals (list of Tensors): Length B list containing regional proposal boxes
        """
        # Step A: Extract shared global feature maps
        feature_maps = self.backbone(images)

        print(
            "Feature maps shape:", feature_maps.shape
        )  # Debugging output to verify dimensions

        # Step B: Execute RoI Align to crop and normalize regional maps into uniform patches
        # roi_align expects proposals formatted as a single combined tensor prefixed with batch indices
        pooled_features = ops.roi_align(
            feature_maps,
            rpn_proposals,
            output_size=self.pool_size,
            spatial_scale=self.spatial_scale,
            sampling_ratio=2,
            aligned=True,
        )

        print(
            "Pooled features shape (after RoI Align):", pooled_features.shape
        )  # Debugging output to verify dimensions

        # Step C: Flatten patches and pass them through downstream classification and regression predictors
        pooled_features = pooled_features.flatten(start_dim=1)

        print(
            "Flattened pooled features shape:", pooled_features.shape
        )  # Debugging output to verify dimensions
        head_outputs = self.box_head(pooled_features)

        print(
            "Head outputs shape:", head_outputs.shape
        )  # Debugging output to verify dimensions

        class_logits = self.class_predictor(head_outputs)
        bbox_deltas = self.bbox_predictor(head_outputs)

        print(
            "Class logits shape:", class_logits.shape
        )  # Debugging output to verify dimensions
        print(
            "BBox deltas shape:", bbox_deltas.shape
        )  # Debugging output to verify dimensions

        return class_logits, bbox_deltas


# =====================================================================
# 2. RUNTIME INFERENCE & POST-PROCESSING INTERFACE
# =====================================================================


@torch.no_grad()
def run_inference(model, images, rpn_proposals, score_thresh=0.05, nms_thresh=0.5):
    model.eval()
    device = images.device

    # Forward Pass
    class_logits, bbox_deltas = model(images, rpn_proposals)

    # Softmax conversion to get exact predictive probability confidence
    probs = torch.softmax(class_logits, dim=-1)

    final_results = []
    proposal_offset = 0

    for img_idx, proposals in enumerate(rpn_proposals):
        num_props = proposals.shape[0]
        if num_props == 0:
            final_results.append(
                {
                    "boxes": torch.empty((0, 4), device=device),
                    "scores": torch.empty((0,), device=device),
                    "labels": torch.empty((0,), dtype=torch.long, device=device),
                }
            )
            continue

        # Isolate slices matching current batch slice
        img_probs = probs[proposal_offset : proposal_offset + num_props]
        img_deltas = bbox_deltas[proposal_offset : proposal_offset + num_props]
        proposal_offset += num_props

        # Exclude background class (assumed to be the last index column)
        fg_probs = img_probs[:, :-1]
        img_scores, img_labels = torch.max(fg_probs, dim=-1)

        # Filter background predictions early via thresholding
        keep_mask = img_scores > score_thresh
        scores = img_scores[keep_mask]
        labels = img_labels[keep_mask]
        filtered_proposals = proposals[keep_mask]
        filtered_deltas = img_deltas[keep_mask]

        if filtered_proposals.shape[0] == 0:
            final_results.append(
                {
                    "boxes": torch.empty((0, 4), device=device),
                    "scores": torch.empty((0,), device=device),
                    "labels": torch.empty((0,), dtype=torch.long, device=device),
                }
            )
            continue

        # Coordinate Delta Decoding
        pw = filtered_proposals[:, 2] - filtered_proposals[:, 0]
        ph = filtered_proposals[:, 3] - filtered_proposals[:, 1]
        px = filtered_proposals[:, 0] + 0.5 * pw
        py = filtered_proposals[:, 1] + 0.5 * ph

        gather_idx = labels * 4
        dx = filtered_deltas[torch.arange(len(labels)), gather_idx]
        dy = filtered_deltas[torch.arange(len(labels)), gather_idx + 1]
        dw = filtered_deltas[torch.arange(len(labels)), gather_idx + 2]
        dh = filtered_deltas[torch.arange(len(labels)), gather_idx + 3]

        nx = pw * dx + px
        ny = ph * dy + py
        nw = pw * torch.exp(dw)
        nh = ph * torch.exp(dh)

        decoded_boxes = torch.stack(
            [nx - 0.5 * nw, ny - 0.5 * nh, nx + 0.5 * nw, ny + 0.5 * nh], dim=1
        )

        # Multiclass Batched NMS to separate independent overlap boundaries safely
        keep_nms = ops.batched_nms(
            decoded_boxes, scores, labels, iou_threshold=nms_thresh
        )

        final_results.append(
            {
                "boxes": decoded_boxes[keep_nms],
                "scores": scores[keep_nms],
                "labels": labels[keep_nms],
            }
        )

    return final_results


# =====================================================================
# 3. VERIFICATION EXECUTION BLOCK
# =====================================================================

if __name__ == "__main__":
    # Settings
    NUM_CLASSES = 3  # Say: 0=Person, 1=Bicycle, 2=Car
    BATCH_SIZE = 2
    IMG_H, IMG_W = 224, 224
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running validation setup on environment device target: {device}")

    # Initialize model
    detector = TwoStageRoIDetector(num_classes=NUM_CLASSES).to(device)

    # Generate mock inputs
    # 1. Batch normalized random images
    mock_images = torch.randn(BATCH_SIZE, 3, IMG_H, IMG_W, device=device)

    # 2. Mock RPN proposals (x1, y1, x2, y2 format in absolute scale mapping bounds)
    mock_proposals_img0 = torch.tensor(
        [
            [10.0, 20.0, 80.0, 90.0],
            [15.0, 22.0, 85.0, 95.0],  # Heavy overlap with box above
            [100.0, 110.0, 200.0, 210.0],
        ],
        dtype=torch.float32,
        device=device,
    )

    mock_proposals_img1 = torch.tensor(
        [[50.0, 50.0, 150.0, 150.0]], dtype=torch.float32, device=device
    )

    # Pack proposals as a list of tensors (one tensor per image)
    mock_proposals_list = [mock_proposals_img0, mock_proposals_img1]

    # Run complete evaluation cycle
    print("\nExecuting forward tracking pass and NMS post-processing pipeline...")
    detections = run_inference(
        model=detector,
        images=mock_images,
        rpn_proposals=mock_proposals_list,
        score_thresh=0.01,  # Lowered score threshold for random weight initialization
        nms_thresh=0.45,
    )

    # Print out results safely
    for i, output in enumerate(detections):
        print(f"\n--- Results for Image {i} ---")
        print(f"Detected Bounding Boxes shape:  {output['boxes'].shape}")
        print(f"Confidence Scores output:       {output['scores'].cpu().numpy()}")
        print(f"Predicted Class Labels output:  {output['labels'].cpu().numpy()}")
